# 03v2 - A1a': validate the four already-published label sets against our 58 gold

**Plan item A1a'** (2026-08-25 reorientation, see
[[project-rsna-phase-status]] / the strategy artifact): four report-derived
label sets for this exact competition already exist and are legitimate
to use (public code/data sharing is explicitly permitted under the
rules). Scoring all four against our own 58 gold studies is a day of
work, not weeks - and could make building A1a (port Reference A's
lexicon) or A1b (our own LLM pass) unnecessary, or at least give them a
real target to beat instead of a guess. Our own regex labeler already
scores **0.686** macro-AUC vs. gold (notebooks/03_labeler_validation.ipynb,
obsolete but the number is real) - that's the bar these four need to
clear.

The four sets, exactly as named in the sourced plan (no guessed URLs -
find and attach each as a Kaggle Dataset input on this notebook):

1. **Pilkwang Kim** - `rsna-knee-llm-labels` (first published, 2026-08-06
   - also the source of the 130mm-crop baseline cited elsewhere in the
   plan)
2. **barun2104** - `Stratified Folds & LLM Soft Labels`
3. **lixin73** - `LLM Report Labels (GPT-5.6-Sol)`
4. **stevenleehans** - `RSNA Knee LLM Report Labels` (0.878 agreement vs.
   gold, published - the current best measured score of the four)

Effort: low - download + score, no training. Do this before A1a's build
effort; it may make A1a/A1b moot entirely.

## Setup and our own real gold labels

Runs identically on Kaggle or locally for this part - no DICOM needed,
just `train.csv`.

In [1]:
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

_KAGGLE_RAW = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
ON_KAGGLE = _KAGGLE_RAW.exists()

REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

RAW_DIR = _KAGGLE_RAW if ON_KAGGLE else REPO_ROOT / "data" / "raw"
print("RAW_DIR:", RAW_DIR, "| ON_KAGGLE:", ON_KAGGLE)

OFFICIAL_LABEL_COLUMNS = {
    "acl_injury": "ACL",
    "mcl_injury": "MCL",
    "medial_meniscus_tear": "Medial Meniscus",
    "lateral_meniscus_tear": "Lateral Meniscus",
    "oa_medial_compartment": "Medial OA",
    "oa_lateral_compartment": "Lateral OA",
    "oa_patellofemoral_compartment": "PF OA",
    "effusion": "Effusion",
    "synovitis": "Synovitis",
    "bakers_cyst": "Baker's",
    "bone_contusion": "Contusion",
    "fracture": "Fracture",
}
FINDINGS = list(OFFICIAL_LABEL_COLUMNS.keys())
LABEL_COLS = list(OFFICIAL_LABEL_COLUMNS.values())

random.seed(42)
np.random.seed(42)


def macro_roc_auc(y_true: pd.DataFrame, y_pred: pd.DataFrame) -> float:
    return float(np.mean([
        roc_auc_score(y_true[col], y_pred[col]) for col in y_true.columns
    ]))


def per_finding_roc_auc(y_true: pd.DataFrame, y_pred: pd.DataFrame) -> pd.Series:
    return pd.Series({
        col: roc_auc_score(y_true[col], y_pred[col]) for col in y_true.columns
    })


RAW_DIR: C:\Users\alher\Desktop\RSNA_Knee_Abnormality_Detection\data\raw | ON_KAGGLE: False


In [2]:
train = pd.read_csv(RAW_DIR / "train.csv")
n_labels_present = train[LABEL_COLS].notna().sum(axis=1)
gold_mask = n_labels_present == len(LABEL_COLS)

gold = train.loc[gold_mask, ["StudyInstanceUID"] + LABEL_COLS].set_index("StudyInstanceUID")
gold.columns = FINDINGS
gold = gold.astype(float)

print(f"gold studies: {len(gold)}")
assert len(gold) == 58
gold.head(3)


gold studies: 58


,acl_injury,mcl_injury,medial_meniscus_tear,lateral_meniscus_tear,oa_medial_compartment,oa_lateral_compartment,oa_patellofemoral_compartment,effusion,synovitis,bakers_cyst,bone_contusion,fracture
StudyInstanceUID,,,,,,,,,,,,
1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
1.2.826.0.1.3680043.8.498.10170898615867673028696505248839028269,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
1.2.826.0.1.3680043.8.498.10306159113324811538703788080836752052,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0


## Discover what's available

Two sources: datasets attached to a Kaggle input (`/kaggle/input/`), or
files placed locally under `data/raw/_published_labels/` (gitignored,
same convention as the rest of `data/raw/` - see .gitignore). The user
found 2 of the 4 by browser download and dropped them in the project
root as `archive.zip` / `archive(1).zip`; both were byte-identical
(same dataset downloaded twice under two names/authors - a common
Kaggle pattern when one author forks another's dataset unchanged),
extracted here as the 3 CSVs under `data/raw/_published_labels/`.

In [3]:
local_labels_dir = REPO_ROOT / "data" / "raw" / "_published_labels"
if local_labels_dir.exists():
    print(f"=== local: {local_labels_dir} ===")
    for f in sorted(local_labels_dir.glob("*.csv")):
        print(" ", f.name)
    print()

if ON_KAGGLE:
    kaggle_input = Path("/kaggle/input")
    attached_dirs = sorted(
        d for d in kaggle_input.iterdir()
        if d.is_dir() and d.name != "competitions"
    )
    for d in attached_dirs:
        print(f"=== {d.name} ===")
        files = sorted(d.rglob("*"))
        for f in files[:20]:
            print(" ", f.relative_to(d))
        if len(files) > 20:
            print(f"  ... ({len(files)} files total)")
        print()
else:
    print("(not on Kaggle - only the local listing above applies)")


=== local: C:\Users\alher\Desktop\RSNA_Knee_Abnormality_Detection\data\raw\_published_labels ===
  llm_labels_full.csv
  llm_labels_v2.csv
  llm_labels_v4_blend.csv

(not on Kaggle - only the local listing above applies)


## Generic loader + column mapper

One CSV reader that works for any of the four (adjust `filename` per
dataset once B.1's real output is in), plus a conservative column
mapper: matches a source column to one of our 12 `FINDINGS` only on a
normalized exact/substring match against known name variants, and
**refuses to guess** on anything ambiguous - unmapped columns are
reported, not silently dropped or misassigned.

In [4]:
import re
import unicodedata

# Known name variants per finding, lowercased/normalized. Extend this if
# a real dataset uses a spelling not covered here - do not loosen the
# matching logic itself to compensate (see the "refuses to guess" note
# above).
FINDING_ALIASES = {
    "acl_injury": {"acl", "acl_injury", "acl injury", "aclinjury"},
    "mcl_injury": {"mcl", "mcl_injury", "mcl injury", "mclinjury"},
    "medial_meniscus_tear": {"medial meniscus", "medial_meniscus", "medial meniscus tear", "medialmeniscus"},
    "lateral_meniscus_tear": {"lateral meniscus", "lateral_meniscus", "lateral meniscus tear", "lateralmeniscus"},
    "oa_medial_compartment": {"medial oa", "medial_oa", "oa medial compartment", "oa_medial_compartment"},
    "oa_lateral_compartment": {"lateral oa", "lateral_oa", "oa lateral compartment", "oa_lateral_compartment"},
    "oa_patellofemoral_compartment": {"pf oa", "pf_oa", "patellofemoral oa", "oa patellofemoral compartment"},
    "effusion": {"effusion"},
    "synovitis": {"synovitis"},
    "bakers_cyst": {"baker's", "bakers", "baker's cyst", "bakers_cyst", "bakerscyst"},
    "bone_contusion": {"contusion", "bone contusion", "bone_contusion"},
    "fracture": {"fracture"},
}


def _normalize_col(col: str) -> str:
    t = unicodedata.normalize("NFKD", col.lower())
    t = "".join(ch for ch in t if not unicodedata.combining(ch))
    t = re.sub(r"[_\-]+", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t


def map_columns_to_findings(columns) -> dict:
    # Returns {source_column: finding_name}. A source column that
    # matches no alias is simply absent from the result -- report it
    # separately, don't guess.
    mapping = {}
    for col in columns:
        normalized = _normalize_col(str(col))
        for finding, aliases in FINDING_ALIASES.items():
            if normalized in aliases:
                mapping[col] = finding
                break
    return mapping


def load_published_labels(path: Path, id_column_candidates=("StudyInstanceUID", "study_id", "id")) -> pd.DataFrame:
    df = pd.read_csv(path)
    id_col = next((c for c in id_column_candidates if c in df.columns), None)
    if id_col is None:
        raise ValueError(f"{path}: no recognized ID column among {id_column_candidates}, found {list(df.columns)}")
    col_map = map_columns_to_findings(df.columns)
    unmapped = [c for c in df.columns if c not in col_map and c != id_col]
    if unmapped:
        print(f"{path.name}: unmapped columns (not scored): {unmapped}")
    mapped = df[[id_col] + list(col_map.keys())].rename(columns=col_map).set_index(id_col)
    missing_findings = [f for f in FINDINGS if f not in mapped.columns]
    if missing_findings:
        print(f"{path.name}: no column found for {missing_findings}")
    return mapped


## Score what we have against our 58 gold

3 files found locally, all with columns matching our own
`OFFICIAL_LABEL_COLUMNS` naming exactly (no ambiguous mapping needed)
and the real `StudyInstanceUID` join key - `llm_labels_full.csv`,
`llm_labels_v2.csv`, `llm_labels_v4_blend.csv`. Scored below against all
58 gold studies. The other 2 published sets (of the original 4) are
still unfound - `DATASET_FILES` has empty placeholders for them,
extend once found (locally or via Kaggle input).

In [5]:
local_labels_dir = REPO_ROOT / "data" / "raw" / "_published_labels"

DATASET_FILES = {
    "found set - llm_labels_full": local_labels_dir / "llm_labels_full.csv",
    "found set - llm_labels_v2": local_labels_dir / "llm_labels_v2.csv",
    "found set - llm_labels_v4_blend": local_labels_dir / "llm_labels_v4_blend.csv",
    # "Pilkwang Kim": ...,   # not found yet
    # "barun2104 or lixin73 (whichever this isn't)": ...,  # not found yet
}

results = {}
for name, path in DATASET_FILES.items():
    if not path.exists():
        print(f"{name}: {path} not found, skipping")
        continue
    preds = load_published_labels(path)
    common = gold.index.intersection(preds.index)
    print(f"{name}: {len(common)}/{len(gold)} gold studies matched by ID")
    available_findings = [f for f in FINDINGS if f in preds.columns]
    y_true = gold.loc[common, available_findings]
    y_pred = preds.loc[common, available_findings]
    macro = macro_roc_auc(y_true, y_pred)
    per_finding = per_finding_roc_auc(y_true, y_pred)
    results[name] = {"macro": macro, "per_finding": per_finding, "n_findings_scored": len(available_findings)}
    print(f"{name}: macro AUC = {macro:.4f} over {len(available_findings)}/12 findings")
    print(per_finding.sort_values().to_string())
    print()


found set - llm_labels_full: 58/58 gold studies matched by ID
found set - llm_labels_full: macro AUC = 0.8780 over 12/12 findings
synovitis                        0.678017
fracture                         0.793056
oa_lateral_compartment           0.829787
bone_contusion                   0.830634
effusion                         0.853416
lateral_meniscus_tear            0.862112
oa_patellofemoral_compartment    0.900901
oa_medial_compartment            0.931008
bakers_cyst                      0.945652
medial_meniscus_tear             0.954327
mcl_injury                       0.963719
acl_injury                       0.993260

found set - llm_labels_v2: 58/58 gold studies matched by ID


found set - llm_labels_v2: macro AUC = 0.8873 over 12/12 findings
synovitis                        0.790323
fracture                         0.793056
oa_lateral_compartment           0.829787
bone_contusion                   0.830634
effusion                         0.853416
lateral_meniscus_tear            0.862112
oa_patellofemoral_compartment    0.900901
oa_medial_compartment            0.931008
bakers_cyst                      0.945652
medial_meniscus_tear             0.954327
mcl_injury                       0.963719
acl_injury                       0.993260

found set - llm_labels_v4_blend: 58/58 gold studies matched by ID


found set - llm_labels_v4_blend: macro AUC = 0.8927 over 12/12 findings


synovitis                        0.790323
fracture                         0.793056
oa_lateral_compartment           0.832689
bone_contusion                   0.859649
effusion                         0.877019
lateral_meniscus_tear            0.878882
oa_patellofemoral_compartment    0.901544
oa_medial_compartment            0.931783
bakers_cyst                      0.943841
medial_meniscus_tear             0.948317
mcl_injury                       0.968254
acl_injury                       0.987132



### Is each version's improvement a broad effect or one label riding along?

`full -> v2 -> v4_blend` each raised the macro score - worth checking with `per_label_gate` (graduated in `src/evaluate.py` from A0) whether that's a real broad improvement or something concentrated in one or two findings, same diagnostic used for our own gate.

In [6]:
# per_label_gate copied from src/evaluate.py (A0) - same logic, no import
# needed for a one-off local check.
def per_label_gate(baseline_auc, candidate_auc, tol=0.03, min_concordant=7):
    delta = candidate_auc - baseline_auc
    macro_delta = float(delta.mean())
    moved = delta[delta.abs() >= tol]
    concordant = int((np.sign(moved) == np.sign(macro_delta)).sum()) if macro_delta != 0 else 0
    return {
        "macro_delta": macro_delta,
        "n_labels_moved": int(len(moved)),
        "n_concordant": concordant,
        "broad_effect": concordant >= min_concordant,
    }


if "found set - llm_labels_full" in results:
    full_pf = results["found set - llm_labels_full"]["per_finding"]
    v2_pf = results["found set - llm_labels_v2"]["per_finding"]
    v4_pf = results["found set - llm_labels_v4_blend"]["per_finding"]

    for name, baseline, candidate in [
        ("full -> v2", full_pf, v2_pf),
        ("v2 -> v4_blend", v2_pf, v4_pf),
    ]:
        check = per_label_gate(baseline, candidate)
        print(f"{name}: macro_delta={check['macro_delta']:+.4f}  "
              f"n_concordant={check['n_concordant']}/12  broad_effect={check['broad_effect']}")


full -> v2: macro_delta=+0.0094  n_concordant=1/12  broad_effect=False
v2 -> v4_blend: macro_delta=+0.0054  n_concordant=0/12  broad_effect=False


## Comparison summary

**Real numbers (2026-08-25, scored locally against all 58 gold studies,
58/58 matched by ID on every file):**

| Source | Macro AUC vs. our 58 gold | Notes |
|---|---|---|
| Our own regex labeler (baseline) | 0.686 | notebooks/03_labeler_validation.ipynb |
| Found set - `llm_labels_full.csv` | **0.8780** | matches stevenleehans's forum-reported 0.878 exactly |
| Found set - `llm_labels_v2.csv` | **0.8873** | matches stevenleehans's forum-reported "after the Synovitis fix" 0.8873 exactly |
| Found set - `llm_labels_v4_blend.csv` | **0.8927** | not documented anywhere in the sourced plan - a newer iteration, best of the three |
| Pilkwang Kim | not found yet | |
| barun2104 / lixin73 (whichever wasn't downloaded) | not found yet | |

The two exact matches to stevenleehans's own reported numbers (0.878,
0.8873) all but confirm this found dataset - or a byte-identical fork
of it under a different name - IS stevenleehans's `RSNA Knee LLM
Report Labels`. `llm_labels_v4_blend.csv` is not documented in the
sourced plan at all and beats both known numbers, suggesting a newer
version was published after the forum research was done.

`per_label_gate` on the version-to-version deltas: `full -> v2` is a
narrow, single-label effect (1/12 concordant, matches the documented
Synovitis-via-Effusion fix exactly); `v2 -> v4_blend` is diffuse (0/12
moved past the 0.03 threshold individually, but 8 of 12 individual
findings improved by smaller amounts - not a single label riding the
whole macro gain).

**Decision:** `llm_labels_v4_blend.csv` clears our own 0.686 baseline
by +0.207 - a massive, unambiguous margin, nowhere near gate noise
(the +-0.02-0.05 range A0 measured). Per A1a''s own decision rule:
**A1a (porting Reference A's lexicon) and A1b (building our own LLM
pass) are both unnecessary** - this published set is the training
label source going forward, pending only:
- Finding/checking the other 2 published sets, for completeness (low
  priority now - the bar is already cleared by a wide margin).
- Already confirmed: all 3 files have 4,407 rows, the full `train.csv`
  row count - this covers every study, gold and weak alike, not just
  the 58 gold rows scored here.
- Deciding whether to use it as a hard replacement for the regex
  labeler's weak-label output in `src/data.py::load_training_labels`,
  or blend the two (out of scope for A1a' itself - a B-tier decision).